In [2]:
# %%
import eurostat
import pandas as pd
import plotly.express as px
import pycountry

c:\Users\henri_ugzoq54\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
# %%
datasets = ["educ_uoe_lang01"]

data = {}

for ds in datasets:
    print(f"Downloading {ds}...")
    data[ds] = eurostat.get_data_df(ds, flags=False)

In [14]:
import os
print(os.getcwd())

c:\Users\henri_ugzoq54\OneDrive\TransfertDocument09-11-2025\2025_2026\Trento M1 DS\Semester 2\Data Vis\Project\Project Visualisation EF\script


In [5]:
def clean_eurostat(df):
    # rename geo column FIRST
    for c in df.columns:
        if "geo" in c.lower():
            df = df.rename(columns={c: "geo"})

    year_cols = [c for c in df.columns if c.isdigit()]
    id_cols = [c for c in df.columns if c not in year_cols]

    df = df.melt(id_vars=id_cols, var_name="year", value_name="value")

    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df = df.dropna(subset=["value"])

    return df

In [10]:
# --- ENGLISH LEARNING ---
df1 = clean_eurostat(data["educ_uoe_lang01"])

if "unit" in df1.columns:
    df1 = df1[df1["unit"] == "PC"]

if "sex" in df1.columns:
    df1 = df1[df1["sex"] == "T"]

df_final = (
    df1[df1["language"] == "ENG"]
    .groupby(["geo","year"])["value"]
    .mean()
    .reset_index()
    .rename(columns={"value": "learning"})
)

df_final = df_final[df_final["geo"].str.len() == 2]

In [20]:
print(df1["language"].unique()[:20])


['ARA' 'BUL' 'CHI' 'CZE' 'DAN' 'DUT' 'ENG' 'EST' 'FIN' 'FRE' 'GER' 'GLE'
 'GRE' 'HRV' 'HUN' 'ITA' 'JPN' 'LAV' 'LIT' 'MLT']


In [11]:
# %%
# --- CLEAN EUROSTAT (MINIMAL PROPRE) ---

def safe_filter(df):
    if "sex" in df.columns and "T" in df["sex"].unique():
        df = df[df["sex"] == "T"]
    return df

df1 = safe_filter(df1)

# garder % uniquement
if "unit" in df1.columns:
    df1 = df1[df1["unit"] == "PC"]

# nettoyer langues
df1 = df1[~df1["language"].isin(["TOTAL", "UNK", "OTH"])]

# --- VARIABLE CLÉ ---
df_final = (
    df1[df1["language"] == "ENG"]
    .groupby(["geo","year"])["value"]
    .mean()
    .reset_index()
    .rename(columns={"value": "learning"})
)

# garder pays
df_final = df_final[df_final["geo"].str.len() == 2]

In [13]:
print("AFTER FILTER ↓")

print("df1 languages:", df1["language"].unique())

print("df1 shape:", df1.shape)


AFTER FILTER ↓
df1 languages: ['ARA' 'BUL' 'CHI' 'CZE' 'DAN' 'DUT' 'ENG' 'EST' 'FIN' 'FRE' 'GER' 'GLE'
 'GRE' 'HRV' 'HUN' 'ITA' 'JPN' 'LAV' 'LIT' 'MLT' 'POL' 'POR' 'RUM' 'RUS'
 'SLO' 'SLV' 'SPA' 'SWE']
df1 shape: (49706, 7)


In [16]:
# %%
ef = pd.read_csv("../data/efiepi_rankings.csv")

ef = ef.rename(columns={
    "Country": "country",
    "Year": "year",
    "Score": "score"
})

ef["year"] = pd.to_numeric(ef["year"])
ef["score"] = pd.to_numeric(ef["score"])

In [17]:
# %%
def iso2_to_iso3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None

df_final["iso3"] = df_final["geo"].apply(iso2_to_iso3)
df_final = df_final[df_final["iso3"].notna()]

In [18]:
# %%
def country_to_iso2(name):
    try:
        return pycountry.countries.lookup(name).alpha_2
    except:
        return None

mapping = {
    "Czech Republic": "Czechia",
    "Russia": "Russian Federation"
}

ef["country_clean"] = ef["country"].replace(mapping)
ef["iso2"] = ef["country_clean"].apply(country_to_iso2)

ef["iso3"] = ef["iso2"].apply(iso2_to_iso3)
ef = ef[ef["iso3"].notna()]

In [19]:
# %%
ef["fluency_rank"] = ef.groupby("year")["score"].rank(pct=True)

In [20]:
# %%
df = df_final.merge(
    ef[["iso3","year","fluency_rank"]],
    on=["iso3","year"],
    how="left"   # ← IMPORTANT
)

# %%
# STANDARDISATION (clé)
df["learning_z"] = df.groupby("year")["learning"].transform(
    lambda x: (x - x.mean()) / x.std()
)

# GAP CORRECT
df["gap"] = df["fluency_rank"] - df["learning_z"]

In [21]:
df = df[df["fluency_rank"].notna()]
# %%
df = df.sort_values("year")

In [22]:
# %%
fig = px.choropleth(
    df,
    locations="iso3",
    color="gap",
    animation_frame="year",
    hover_name="geo",
    color_continuous_scale="RdBu",
    range_color=[-1,1],   # ← IMPORTANT
    title="Efficiency (standardized): Learning vs Fluency"
)

fig.update_geos(scope="europe")
fig.show()

In [23]:
print(df.shape)
print(df.describe())
print(df.head())
# %%
print(df[["learning","learning_z","fluency_rank","gap"]].describe())

(167, 7)
              year    learning  fluency_rank  learning_z         gap
count   167.000000  167.000000    167.000000  167.000000  167.000000
mean   2020.467066   84.613673      0.623044   -0.064770    0.687814
std       2.296630   12.793793      0.247343    0.856603    0.969235
min    2017.000000   56.280000      0.066667   -2.107522   -0.697399
25%    2018.000000   73.760000      0.421216   -0.707320   -0.028094
50%    2020.000000   86.520000      0.645161    0.030158    0.328519
75%    2022.000000   96.220000      0.836022    0.659652    1.475965
max    2024.000000   99.540000      1.000000    1.019740    2.878024
   geo  year  learning iso3  fluency_rank  learning_z       gap
6   AT  2017     99.54  AUT      0.730769    1.019248 -0.288479
22  BE  2017     56.28  BEL      0.653846   -2.107522  2.761368
34  BG  2017     85.32  BGR      0.346154   -0.008553  0.354707
58  CZ  2017     91.24  CZE      0.423077    0.419336  0.003741
70  DE  2017     72.76  DEU      0.769231   -0.916